# 🔍 Fraud Detection: Exploratory Data Analysis & Feature Engineering

This notebook provides a forensic deep-dive into the credit card fraud transaction dataset.

### Objectives:
1. **Class Imbalance**: Quantify the severe ratio of legitimate to fraudulent transactions.
2. **Amount Analysis**: Compare spending distributions and assess the need for `RobustScaler`.
3. **Cyclical Diurnal Time**: Transform raw elapsed seconds into sinusoidal 24-hour time coordinates.
4. **PCA Latent Space**: Uncover the highest-divergence forensic features ($V_{14}, V_{12}, V_{10}, V_4$).
5. **Model Evaluation & Metric Selection**: Demonstrate why PR-AUC (Precision-Recall AUC) dominates ROC-AUC on imbalanced data.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Visual styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Data Ingestion & Overview
Load the transaction records from `data/raw/creditcard.csv`.

In [ ]:
data_path = Path('../data/raw/creditcard.csv')
if not data_path.exists():
    data_path = Path('data/raw/creditcard.csv')

df = pd.read_csv(data_path)
print(f'Dataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 2. Extreme Class Imbalance
In real-world fraud detection, legitimate transactions constitute >99% of total volume.

In [ ]:
counts = df['Class'].value_counts()
legit_count = counts.get(0, 0)
fraud_count = counts.get(1, 0)
fraud_pct = (fraud_count / len(df)) * 100

print(f'Legitimate (Class 0): {legit_count:,} ({100 - fraud_pct:.2f}%)')
print(f'Fraudulent (Class 1): {fraud_count:,} ({fraud_pct:.3f}%)')
print(f'Imbalance Ratio:      1:{legit_count // max(fraud_count, 1)}')

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=['Legitimate (0)', 'Fraud (1)'], y=[legit_count, fraud_count], palette=['#10b981', '#ef4444'], ax=ax)
ax.set_yscale('log')
ax.set_ylabel('Count (Log Scale)')
ax.set_title('Severe Class Imbalance in Transactions')
plt.tight_layout()
plt.show()

## 3. Transaction Amount Analysis
Compare the monetary profiles of fraudulent charges versus normal consumer spending.

In [ ]:
print('Amount Statistics by Class:')
print(df.groupby('Class')['Amount'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df[df['Class'] == 0]['Amount'], bins=50, color='#10b981', log_scale=True, ax=axes[0])
axes[0].set_title('Legitimate Transactions: Amount (Log Scale)')
axes[0].set_xlabel('Amount (€)')

sns.histplot(df[df['Class'] == 1]['Amount'], bins=50, color='#ef4444', log_scale=True, ax=axes[1])
axes[1].set_title('Fraudulent Transactions: Amount (Log Scale)')
axes[1].set_xlabel('Amount (€)')
plt.tight_layout()
plt.show()

## 4. Feature Engineering: Cyclic Diurnal Time Transform
The raw `Time` feature is monotonic elapsed seconds from the start of the recording.
We convert this into diurnal cycle features using sinusoidal encoding:
$$\text{time\_day} = \text{Time} \pmod{86400}$$
$$\sin\_\text{time} = \sin\left(\frac{2\pi \cdot \text{time\_day}}{86400}\right), \quad \cos\_\text{time} = \cos\left(\frac{2\pi \cdot \text{time\_day}}{86400}\right)$$

In [ ]:
seconds_in_day = 86400.0
time_day = df['Time'] % seconds_in_day
sin_time = np.sin(2 * np.pi * time_day / seconds_in_day)
cos_time = np.cos(2 * np.pi * time_day / seconds_in_day)

fig, ax = plt.subplots(figsize=(6, 6))
scatter = ax.scatter(sin_time[:1000], cos_time[:1000], c=time_day[:1000] / 3600, cmap='twilight', alpha=0.7, s=15)
ax.set_title('24-Hour Diurnal Clock (Sin/Cos Cyclical Projection)')
ax.set_xlabel('sin(Time)')
ax.set_ylabel('cos(Time)')
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Hour of Day (0-24)')
plt.tight_layout()
plt.show()

## 5. Latent PCA Anomaly Dimensions
Features $V_1 \dots V_{28}$ are PCA-transformed latent behavioral signatures.
We examine correlation with fraud class to find the most discriminative dimensions.

In [ ]:
corrs = df[[f'V{i}' for i in range(1, 29)] + ['Class']].corr()['Class'].drop('Class')
top_negative = corrs.sort_values().head(5)
top_positive = corrs.sort_values(ascending=False).head(5)

top_features = pd.concat([top_negative, top_positive])
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#ef4444' if x < 0 else '#3b82f6' for x in top_features.values]
sns.barplot(x=top_features.index, y=top_features.values, palette=colors, ax=ax)
ax.set_title('Top 10 Forensic Features Correlated with Fraud')
ax.set_ylabel('Pearson Correlation with Class')
plt.tight_layout()
plt.show()

## 6. Model Training & Precision-Recall Comparison
We compare Logistic Regression (with balanced class weights) against XGBoost (with `scale_pos_weight`).
PR-AUC is our primary metric because it focuses exclusively on the positive (fraud) class.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score
from xgboost import XGBClassifier

# Feature matrix
X = df.copy()
X['sin_time'] = np.sin(2 * np.pi * (X['Time'] % 86400) / 86400)
X['cos_time'] = np.cos(2 * np.pi * (X['Time'] % 86400) / 86400)
X['scaled_amount'] = RobustScaler().fit_transform(X[['Amount']])

feature_cols = [f'V{i}' for i in range(1, 29)] + ['sin_time', 'cos_time', 'scaled_amount']
y = X['Class'].values
X_train, X_test, y_train, y_test = train_test_split(X[feature_cols], y, test_size=0.2, random_state=42, stratify=y)

# Logistic Regression Baseline
lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]

# XGBoost with scale_pos_weight
ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb = XGBClassifier(scale_pos_weight=ratio, max_depth=4, learning_rate=0.05, n_estimators=100, random_state=42, eval_metric='aucpr')
xgb.fit(X_train, y_train)
xgb_probs = xgb.predict_proba(X_test)[:, 1]

# PR-AUC curves
lr_precision, lr_recall, _ = precision_recall_curve(y_test, lr_probs)
xgb_precision, xgb_recall, _ = precision_recall_curve(y_test, xgb_probs)

lr_prauc = average_precision_score(y_test, lr_probs)
xgb_prauc = average_precision_score(y_test, xgb_probs)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(lr_recall, lr_precision, label=f'Logistic Regression (PR-AUC = {lr_prauc:.3f})', color='#3b82f6', lw=2)
ax.plot(xgb_recall, xgb_precision, label=f'XGBoost (PR-AUC = {xgb_prauc:.3f})', color='#ef4444', lw=2)
ax.set_title('Precision-Recall Curve Comparison')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

print(f'Logistic Regression PR-AUC: {lr_prauc:.4f}')
print(f'XGBoost PR-AUC:             {xgb_prauc:.4f}')

## Summary & Conclusions
* **Class Imbalance**: Gradient reweighting (`scale_pos_weight`) effectively penalizes false negatives without generating artificial noisy interpolation points like SMOTE.
* **Cyclical Features**: Projecting transaction time onto $\sin / \cos$ dimensions preserves time-of-day continuity across midnight boundaries.
* **Forensic Signals**: Components $V_{14}, V_{12}, V_{10}$ offer the strongest negative anomaly shifts for transaction fraud, matching what our Explainable AI (TreeSHAP) engine surfaces to investigators in the dashboard.